# Structure Unstructured Restaurant Data with an LLM

## Objectives

- Load raw restaurant descriptions and review data from unstructured TXT files
- Use a multimodal/LLM-based pipeline to extract structured attributes (e.g., cuisine type, ambiance, dietary options, and signature dishes)
- Convert the extracted information into a well-defined JSON schema suitable for indexing and search

```bash
%%capture
%pip install numpy==2.3.4
%pip install matplotlib==3.10.7
%pip install openai
```

### Import required libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from openai import OpenAI
import time

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

File is loaded from the local directory. If you are running this notebook in a different environment, you may download the file using the following command:
```python
!curl -O https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt # For windows systems
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/1r_mM6ZPYNxcFv65QkzubA/California-Culinary-Map.txt # For linux systems
```

### Step 1: Load the data

In [2]:
file_path = "California-Culinary-Map.txt"
with open(file_path, "r") as f:
    data = f.read()
print(f"Data loaded has {len(data)} characters.")
print("Printing first 300 characters:")
print(data[:300])
print("Data loaded successfully.")

Data loaded has 74679 characters.
Printing first 300 characters:
### The Culinary Map of California

**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-
Data loaded successfully.


### Step 2: Split the restaurant paragraphs into a Python list

In [3]:
restaurant_list = data.split("\n\n")
restaurant_list = restaurant_list[1:] # removing the first element which is dataset name
print(f"Number of restaurants in the list: {len(restaurant_list)}")
print(restaurant_list[0]) # printing the first record for closer look

Number of restaurants in the list: 210
**The Gilded Artichoke** brings a **bohemian chic** energy to the hills of **Silver Lake**, operating as an **upscale bistro** that prioritizes **Farm-to-Table Californian** ingredients. The space feels like a high-end greenhouse with its reclaimed wood and floor-to-ceiling windows, perfectly complementing the **4.5/5** rating earned by its lavender-rubbed roasted chicken and delicate heirloom tomato tarts.  Price range: $$$$


### Implement the model function

In [4]:
def llm_model(system_msg, prompt_txt):
    # Point client to local Ollama server
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"  # Required by SDK, but ignored by Ollama
    )
    # Defining the message for LLM
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_txt} 
    ]
    model_name = "llama3.2"
    # Final response
    response = client.chat.completions.create(
        messages = messages, 
        model=model_name
        )
    output_text = response.choices[0].message.content
    return output_text
def safe_llm_call(system_msg, prompt_txt, retries=3):
    for attempt in range(retries):
        try:
            return llm_model(system_msg, prompt_txt)
        except Exception as e:
            time.sleep(2)
    return "Failed after retries:"

### Testing the LLM function

In [5]:
system_msg = "You are a helpful assistant."
prompt_txt = "Which place is warmer in winter? Hawaii or Greenland?"
print(safe_llm_call(system_msg, prompt_txt))

In winter, Hawaii is significantly warmer than Greenland.

Hawaii is a tropical island chain located in the Pacific Ocean, and its average winter temperatures range from the mid-70s to mid-80s Fahrenheit (23-30°C).

On the other hand, Greenland is a subarctic island in the North Atlantic Ocean, known for its cold and icy climate. During the winter months, average temperatures in Greenland can range from -13°C to -22°C (9-8°F), with even colder temperatures in the interior and northern parts of the island.

The temperature difference between Hawaii and Greenland during winter is quite dramatic, making Hawaii a much warmer destination in the winter months.


### Define the template

- Implement a function that generates prompts for the LLM, given an input restaurant description paragraph.
- Design a prompt template that enables the model to consistently transform *any* input restaurant description into the required structured JSON format.
- Apply the **one-shot prompting** technique, which includes a single example of the desired output within the prompt to guide the model’s response.

**Note:** For the price range field, instruct the LLM to convert dollar signs (e.g., \\\\\\\\\\\\, \$\$, $$$) into an integer representing the number of dollar symbols.

In [6]:
EXAMPLE_RESTAURANT_PARAGRAPH = restaurant_list[1] # using the second entry as an example
EXAMPLE_OUTPUT = """
    {{
    "name": "Mar de Cortez",
    "location": "Santa Monica",
    "type": "casual taqueria",
    "food_style": "Baja-style seafood",
    "rating": 4.2,
    "price_range": 1,
    "signatures": [
        "beer-battered snapper tacos",
        "zesty octopus ceviche"
    ],
    "vibe": "salt-air energy",
    "environment": "a premier sun-drenched spot for open-air dining near the pier."
    "shortcomings": []
    }}
"""
# Design the prompt template
def restaurant_data_structure_prompt_generation(restaurant_paragraph):
    base_system_msg = f"""
     You are an expert data extraction assistant. 
     Your job is to extract structured information about restaurants from unstructured text and convert it into pure, valid JSON format.

     Guiding Rules:
    1. Output strictly valid JSON with no trailing commas or missing quotes.
    2. Do NOT wrap the output in markdown code fences (e.g., no ```json ... ```).
    3. Do NOT include any conversational filler, intro, or explanation outside of the JSON object.
    4. Maintain the exact field names and types specified in the instructions and example.
    """
    base_user_prompt = f"""
    Task:
    Extract information from the restaurant description below and format it into a valid JSON object with the following schema:
    - "name": string
    - "location": string
    - "type": string
    - "food_style": string
    - "rating": number (float or int)
    - "price_range": number (integer representing scale/cost tier)
    - "signatures": list of strings (key dishes/drinks)
    - "vibe": string
    - "environment": string
    - "shortcomings": list of strings (complaints, cons, or missing items; empty list [] if none mentioned)
    
    Restaurant_Description:
    {restaurant_paragraph}

    Example:
    Input restaurant description:{EXAMPLE_RESTAURANT_PARAGRAPH}
    Output: {EXAMPLE_OUTPUT}
    """
    return base_system_msg, base_user_prompt



### Testing the prompt template

In [7]:
restaurant_paragraph = restaurant_list[0]
base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph = restaurant_paragraph)
test_response = safe_llm_call(system_msg = base_system_msg, prompt_txt = base_user_prompt)
print("Test Response:\n", test_response)

Test Response:
 {
"name": "The Gilded Artichoke",
"location": "Silver Lake",
"type": "upscale bistro",
"food_style": "Farm-to-Table Californian",
"rating": 4.5,
"price_range": 3,
"signatures": [
  "lavender-rubbed roasted chicken",
  "delicate heirloom tomato tarts"
],
"vibe": "bohemian chic",
"environment": "a high-end greenhouse with reclaimed wood and floor-to-ceiling windows",
"shortcomings": []
}


### Validation of the LLM output


In [8]:
# Validation
from pydantic import BaseModel, ValidationError, Field
from typing import List, Optional
# Defining the schema
class Restaurant(BaseModel):
    name: str
    location: str
    type: str
    food_style: str
    rating: Optional[float] = None
    price_range: Optional[int] = None
    signatures: List[str] = Field(default_factory=list)
    vibe: Optional[str] = None
    environment: str
    shortcomings: List[str] = Field(default_factory=list)

# Using the validation method to validate the test response
try:
    restaurant_data = Restaurant.model_validate_json(test_response)
    print(f"Success! Validated: {restaurant_data.name}")
except:
    print(f"Validation failed: {e.json()}")

Success! Validated: The Gilded Artichoke


### Define prompts to have an LLM that auto repairs outputs to JSON format

The previously defined a schema to validate the output alone does not fix incorrect results. To address this, introducing an additional LLM “expert” whose sole responsibility is to automatically repair and correct outputs that do not conform to the required JSON format.

In this step, I am defineing the prompt template function for this LLM.  This function takes:

- `failed_json_output`: The failed json output
- `error_message`: The error message generated by the schema validation if a mistake is detected

Prompting to fee the LLM:

- The original wrong output by feeding `failed_json_output`
- The guidance on correction by feeding `error_message`

In [9]:
def JSON_repair_prompt(failed_json_output, error_message):
    auto_repair_message = f"""
    You are an expert JSON repair assistance. Your sole task is to fix malform or invalid JSON strings.
    - Output ONLY valid , parseable JSON code.
    - Do NOT wrap the output in markdown code blocks (e.g., no '''json...''').
    - Do NOT include any explanation, introduction or conversational text.
    - Preserve all original keys, values, and data structure as closely as possible.
    """
    auto_repair_prompt = f"""
    The following JSON string failed to parse:
    
    ---FAILED JSON STRING ---
    {failed_json_output}

    ---PARSER ERROR MESSAGE ---
    {error_message}
    
    Please fix the syntax error(s) and return ONLY the corrected, valid JSON string.
    """

    return auto_repair_message, auto_repair_prompt

### Run the for loop to go over all the restaurant data in the list

In [10]:
structured_restaurant_lists = []
for i, restaurant_paragraph in enumerate(restaurant_list):
    # Production the inital output
    base_system_msg, base_user_prompt = restaurant_data_structure_prompt_generation(restaurant_paragraph = restaurant_paragraph)
    response_json_output = safe_llm_call(system_msg=base_system_msg, prompt_txt=base_user_prompt)
    # Validating the auto correction loop
    for attempt in range(3):
        try:
            restaurant_data = Restaurant.model_validate_json(response_json_output)
            is_valid = True
        except ValidationError as e:
            attempt += 1
            error_message = e.json()
            repair_system_msg, repair_user_prompt = JSON_repair_prompt(response_json_output, error_message= error_message)
            response_json_output = safe_llm_call(system_msg=repair_system_msg, prompt_txt=repair_user_prompt)

    # Appending the finalized response to the structured restaurant list
    structured_restaurant_lists.append(restaurant_data if is_valid else response_json_output)
    # A manual progress bar
    if (i+1)%20 == 0:
        print(f'{i+1} out of {len(restaurant_list)} is done')
# A final message to notify the completion
print('ALL DONE!!')

20 out of 210 is done
40 out of 210 is done
60 out of 210 is done
80 out of 210 is done
100 out of 210 is done
120 out of 210 is done
140 out of 210 is done
160 out of 210 is done
180 out of 210 is done
200 out of 210 is done
ALL DONE!!


In [11]:
# Print the 50th item in the structured_restaurant_lists
structured_restaurant_lists[49]

Restaurant(name='The Neon Noodle Bar', location='Monterey Park', type='cafe', food_style='Hong Kong', rating=4.0, price_range=0, signatures=['instant noodle gourmet bowls', 'pineapple buns with cold butter'], vibe='high-energy', environment='', shortcomings=[])

In [12]:
structured_restaurant_lists[169]

Restaurant(name='The Velvet Poppy', location='San Luis Obispo', type='bistro', food_style='eclectic', rating=4.5, price_range=2, signatures=['tri-tip sliders', 'blackberry-infused salads'], vibe='charming and eclectic', environment='a historic brick alleyway', shortcomings=[])

# Saving the output file for backup

In [ ]:
structured_restaurant_lists_json = [
    restaurant.dict() for restaurant in structured_restaurant_lists
]
#For each item in the restaurant list, assign it with an itemId to be consistent with the one in the user review data:
for i, response in enumerate(structured_restaurant_lists_json):
    response['itemId'] = 1000001 + i
    structured_restaurant_lists_json[i] = response
    
filename = 'structured_restaurant_data.json'
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(structured_restaurant_lists_json, f, indent=4)